# Prepare two-reference drift inputs

Build the exact `Y,L,lat,lon` references consumed by `5a_refactor_drift_analysis.ipynb`. The production matrix is May and November initialization for `TREFHT` and `SST`, for both Reanalysis and JRA55_FOSIRL. Each product contains observations (`X_obs`), the E3SM-LE historical ensemble-mean attractor (`X_att`), and its ensemble spread (`sigma_att`). Monthly `L=1..24` caches are required; seasonal caches are never substituted.

In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np
import xarray as xr
import esp_lab

REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics import two_reference_drift as drift_reference_workflow
drift_reference_workflow = importlib.reload(drift_reference_workflow)

atomic_to_netcdf = drift_reference_workflow.atomic_to_netcdf
discover_monthly_hindcast = drift_reference_workflow.discover_monthly_hindcast
expand_requests = drift_reference_workflow.expand_requests
load_drift_references = drift_reference_workflow.load_drift_references
prepare_monthly_hindcast_cache = drift_reference_workflow.prepare_monthly_hindcast_cache
prepare_reference_product = drift_reference_workflow.prepare_reference_product
reference_output_path = drift_reference_workflow.reference_output_path
select_initialization_years = drift_reference_workflow.select_initialization_years

## Explicit reference configuration

ERA5 `tas` supplies TREFHT observations and HadISST2 `sst` supplies SST observations. The independent model reference is the E3SM-LE ensemble mean; its ensemble spread supplies the normalization scale. All three fields are matched to each hindcast's exact valid times and grid.

E3SM monthly means may be timestamped at the end of their averaging interval. The shared workflow uses `time_bnds[:,0]` when available and applies the project's mid-month convention before constructing calendar-month climatologies.

`FORCE_REFERENCES` controls only the small prepared-reference products. Missing `Y,M,L=1..24` hindcast caches are reported in a preflight and built only when `BUILD_MISSING_MONTHLY_CACHES=True`; cache construction reads the full requested initialization/member matrix and can take substantially longer than rebuilding references.

In [2]:
ARCHIVE_ROOT = Path('/global/cfs/cdirs/e3sm/S2S2D/E3SMLE')
OBS_ROOT = Path('/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series')
POST_PROCESS_ROOT = Path('/global/cfs/cdirs/e3sm/S2S2D/post_process')
CLIMATOLOGY_YEARS = (1981, 2010)
ANALYSIS_YEARS = (1980, 2017)
INIT_MONTHS = (5, 11)
SOURCES = ('Reanalysis', 'JRA55_FOSIRL')
FORCE_REFERENCES = False
BUILD_MISSING_MONTHLY_CACHES = True
INIT_YEARS = list(range(ANALYSIS_YEARS[0], ANALYSIS_YEARS[1] + 1))
MEMBERS = [f'EN{i:02d}' for i in range(10)]
CASE_PREFIXES = {
    'Reanalysis': 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce',
    'JRA55_FOSIRL': 'WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL',
}
VARIABLE_CONFIG = {
    'TREFHT': {
        'component': 'atm',
        'hindcast_variable': 'TREFHT',
        'observation_path': OBS_ROOT / 'ERA5/tas_197901_201912.nc',
        'observation_variable': 'tas',
        'historical_variable': 'TREFHT',
    },
    'SST': {
        'component': 'ocn',
        'hindcast_variable': 'timeMonthly_avg_activeTracers_temperature',
        'observation_path': OBS_ROOT / 'HadISST2/sst_186901_202212.nc',
        'observation_variable': 'sst',
        'historical_variable': 'timeMonthly_avg_surface_temperature',
    },
}

for variable, config in VARIABLE_CONFIG.items():
    component = config['component']
    archive_variable = config['historical_variable']
    relative = Path(component) / 'ts/monthly/180x360_aave/1yr'
    config['mean_paths'] = sorted((ARCHIVE_ROOT / 'ensmean/post' / relative).glob(f'{archive_variable}_*.nc'))
    config['spread_paths'] = sorted((ARCHIVE_ROOT / 'ensspread/post' / relative).glob(f'{archive_variable}_*.nc'))

REQUESTS = [
    {
        'sources': SOURCES, 'component': config['component'],
        'variables': [variable], 'init_months': INIT_MONTHS,
        'hindcast_variable': config['hindcast_variable'], 'regrid': True,
    }
    for variable, config in VARIABLE_CONFIG.items()
]
JOBS = expand_requests(REQUESTS)

In [ ]:
def validate_monthly_inventory(paths, variable, label):
    if not paths:
        raise FileNotFoundError(f'No files found for {label}')
    with xr.open_mfdataset(
        paths, combine='by_coords', chunks={}, data_vars='minimal',
        coords='minimal', compat='override',
    ) as dataset:
        if variable not in dataset:
            raise KeyError(f'{variable!r} not found in {label}; available={list(dataset.data_vars)}')
        bounds_name = dataset.time.attrs.get('bounds')
        if bounds_name in dataset:
            bounds = dataset[bounds_name]
            bounds_dims = [dim for dim in bounds.dims if dim not in dataset.time.dims]
            if len(bounds_dims) != 1:
                raise ValueError(
                    f'{bounds_name!r} must have exactly one bounds dimension; '
                    f'found dimensions {bounds.dims}'
                )
            represented = bounds.isel({bounds_dims[0]: 0})
        else:
            represented = dataset.time
        years = np.asarray(represented.dt.year.values, dtype=int)
        months = np.asarray(represented.dt.month.values, dtype=int)
    actual = set(zip(years.tolist(), months.tolist()))
    expected = {(year, month) for year in range(CLIMATOLOGY_YEARS[0], CLIMATOLOGY_YEARS[1] + 1) for month in range(1, 13)}
    missing = sorted(expected - actual)
    if missing:
        preview = ', '.join(f'{year:04d}-{month:02d}' for year, month in missing[:12])
        raise RuntimeError(f'{label} is missing {len(missing)} climatology months: {preview}')
    print(f'{label}: {len(paths)} files, {years.min()}-{years.max()}')

for variable, config in VARIABLE_CONFIG.items():
    if not config['observation_path'].is_file():
        raise FileNotFoundError(config['observation_path'])
    validate_monthly_inventory(config['mean_paths'], config['historical_variable'], f'{variable} ensemble mean')
    validate_monthly_inventory(config['spread_paths'], config['historical_variable'], f'{variable} ensemble spread')

available_caches = {}
missing_cache_jobs = []
for job in JOBS:
    key = (job['source'], job['init_month'], job['variable'])
    try:
        available_caches[key] = discover_monthly_hindcast(job)
    except FileNotFoundError:
        missing_cache_jobs.append(job)
print(f'Monthly-cache preflight: {len(available_caches)} available, {len(missing_cache_jobs)} missing')
for job in missing_cache_jobs:
    print(f"  missing: {job['source']} {job['variable']} init={job['init_month']:02d}")

if missing_cache_jobs and not BUILD_MISSING_MONTHLY_CACHES:
    raise FileNotFoundError('Monthly hindcast caches are missing; see the preflight list above')

for job in JOBS:
    key = (job['source'], job['init_month'], job['variable'])
    if key in available_caches:
        hindcast = available_caches[key]
    else:
        if not BUILD_MISSING_MONTHLY_CACHES:
            raise
        print(f"Building monthly cache: {job['source']} {job['variable']} init={job['init_month']:02d}")
        prepare_monthly_hindcast_cache(
            job, case_prefix=CASE_PREFIXES[job['source']], init_years=INIT_YEARS,
            members=MEMBERS, data_root=POST_PROCESS_ROOT, lead_count=24,
            grid='180x360_aave', frequency='monthly', chunk='2yr',
            chunks={'Y': 3, 'L': 24, 'M': 2, 'lat': 90, 'lon': 180}, force=False,
        )
        hindcast = discover_monthly_hindcast(job)
    print(f"Monthly hindcast: {job['source']} {job['variable']} {job['init_month']:02d} -> {hindcast}")

In [ ]:
reference_paths = {}
for job in JOBS:
    config = VARIABLE_CONFIG[job['variable']]
    destination = reference_output_path(job)
    rebuild = FORCE_REFERENCES or not destination.exists()
    stale_reason = 'forced rebuild' if FORCE_REFERENCES else 'missing product'
    if destination.exists() and not FORCE_REFERENCES:
        try:
            with load_drift_references(
                job['source'], job['init_month'], job['variable'],
                component=job['component'], path=destination,
                require_attractor_spread=True,
            ) as existing:
                expected_attrs = {
                    'attractor_climatology_period': f'{CLIMATOLOGY_YEARS[0]}-{CLIMATOLOGY_YEARS[1]}',
                    'analysis_start_year': ANALYSIS_YEARS[0],
                    'analysis_end_year': ANALYSIS_YEARS[1],
                }
                mismatches = {
                    name: (existing.attrs.get(name), value)
                    for name, value in expected_attrs.items()
                    if existing.attrs.get(name) != value
                }
                rebuild = bool(mismatches)
                stale_reason = f'metadata mismatch: {mismatches}' if mismatches else ''
        except (KeyError, OSError, ValueError) as error:
            rebuild = True
            stale_reason = str(error)
        status = f'Rebuilding stale ({stale_reason})' if rebuild else 'Exists and validates'
        print(f'{status}: {destination}')
    if rebuild:
        references = prepare_reference_product(
            job, observation_path=config['observation_path'],
            historical_path=config['mean_paths'],
            observation_variable=config['observation_variable'],
            historical_variable=config['historical_variable'],
            spread_path=config['spread_paths'],
            spread_variable=config['historical_variable'],
            climatology_years=CLIMATOLOGY_YEARS, analysis_years=ANALYSIS_YEARS,
        )
        atomic_to_netcdf(references, destination)
        print(f'Saved: {destination}')
    reference_paths[(job['source'], job['init_month'], job['variable'])] = destination

    hindcast_path = discover_monthly_hindcast(job)
    with xr.open_dataset(hindcast_path, chunks={}) as hindcast, load_drift_references(
        job['source'], job['init_month'], job['variable'],
        component=job['component'], path=destination, hindcast=None,
        require_attractor_spread=True,
    ) as prepared:
        expected_time = select_initialization_years(hindcast.time, ANALYSIS_YEARS)
        np.testing.assert_array_equal(prepared.Y, expected_time.Y)
        np.testing.assert_array_equal(prepared.L, hindcast.L)
        np.testing.assert_array_equal(prepared.valid_time, expected_time)
        np.testing.assert_array_equal(prepared.lat, hindcast.lat)
        np.testing.assert_array_equal(prepared.lon, hindcast.lon)
        assert set(('X_obs', 'X_att', 'sigma_att')) <= set(prepared.data_vars)
        assert prepared.X_obs.attrs['units'] == prepared.X_att.attrs['units'] == prepared.sigma_att.attrs['units']
        assert not bool((prepared.sigma_att < 0).any())
        print(f"Validated 5a interface: {job['variable']} init={job['init_month']:02d} {dict(prepared.sizes)} {prepared.X_obs.attrs['units']}")
reference_paths

Rebuilding stale (Prepared drift reference lacks the required historical ensemble spread (sigma_att): /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_05_TREFHT_monthly_references.nc. Rebuild it with jupyter/0_run_drift_references.ipynb.): /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_05_TREFHT_monthly_references.nc
Saved: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_05_TREFHT_monthly_references.nc
Validated 5a interface: TREFHT init=05 {'Y': 38, 'L': 24, 'lat': 180, 'lon': 360} degC
Saved: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_11_TREFHT_monthly_references.nc
Validated 5a interface: TREFHT init=11 {'Y': 38, 'L': 24, 'lat': 180, 'lon': 360} degC
Rebuilding stale (Prepared drift reference lacks the required historical ensemble spread (sigma_att): /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/JRA55_FOS

{('Reanalysis',
  5,
  'TREFHT'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_05_TREFHT_monthly_references.nc'),
 ('Reanalysis',
  11,
  'TREFHT'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/Reanalysis_11_TREFHT_monthly_references.nc'),
 ('JRA55_FOSIRL',
  5,
  'TREFHT'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/JRA55_FOSIRL_05_TREFHT_monthly_references.nc'),
 ('JRA55_FOSIRL',
  11,
  'TREFHT'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/TREFHT/JRA55_FOSIRL_11_TREFHT_monthly_references.nc'),
 ('Reanalysis',
  5,
  'SST'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/SST/Reanalysis_05_SST_monthly_references.nc'),
 ('Reanalysis',
  11,
  'SST'): PosixPath('/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/drift_diagnostics/references/SST/Reanalysis_11_SST_monthly_references.nc'),
 ('JRA55_FOSIRL',

: 